## preprocess

- for pdf from HAL archiv , remove first page (10min)

(1-2h)
- [x] extract and keep aside 
    - FR/EN abstract from EN/FR texts
    - refs/author list
    - figure-related ocr texts
- [x] remove page numbers / headers (record for files with these information, to run it automatically next time)
- [x] move footnote to the end of the page

- [x] check and correct manually, e.g.
    - footnotes that spread several pages
    - merge split text blocks which come from the same paragraph
    - checke damaged unicode char ￿ 
> later : match first sections, remove orphan sections


(30min)
- identify sections, record aside the information
- extract and keep aside table, record their position (in paragraph)


- to segment

 à noter : some equations are inserted as picture and omitted in our extraction

In [1]:
import re
import os
from pathlib import Path
import unicodedata
import pypandoc
import pandas as pd

In [ ]:
# processed before extracting references
# F12-1083 en
# F12-1092
# taln2013-long-009


# taln2013-long-002 fr

# taln-2010-long-040 fr

# taln-2010-long-025 en
# taln-2007-poster-009 en
# F14-107 en


In [5]:

# to check special symbol: 
# F04-25    ð

# ~~_n_~~ _bEnfant_ 
# ~~1~~

# check encodings like this: ~~_n_~~

#     ￿  taln-2013-long-009
# algo: F14-16 fr line 109

# F14-38

# ￿
# to extract tables: 

#     taln-2007-poster-009
# TAB. 1 

['shdfbywsf']

In [ ]:
STORE_DIR1 = f"{DATA_DIR}/processed_md"

for docid in os.listdir(STORE_DIR1):
    if docid in ['.DS_Store']:
        continue

    print(docid)
    for lang in ['en', 'fr']:
        md_path = f"{STORE_DIR1}/{docid}/document_{lang}.md"

        if not os.path.exists(md_path):
            print(f"File not found : {md_path}")
            continue
            
        md_txt = open(md_path, 'rt').read()
        # md_txt = move_footnotes_to_end(md_txt)
        # md_txt = remove_fig_txt(md_txt)
        # md_txt = fix_latex_accents(md_txt)
        md_txt = remove_special_symbol_stricks(md_txt)
        
        to_write = md_txt
        with open(md_path, 'wt', encoding = 'utf-8') as f:
            f.write(to_write)

In [118]:
# for pdf from HAL archiv , remove first page (10min)

def remove_hal_page(txt):
    hal_descrip = "is a multi-disciplinary open access archive for the deposit and dissemination of scientific research documents, whether they are published or not. The documents may come from teaching and research institutions in France or abroad, or from public or private research centers"
    res = txt.split('HAL Authorization')
    if len(res)> 1:
        assert(len(res) == 2)
        assert(hal_descrip in res[0])
        assert(hal_descrip not in res[1])
        
        res = res[1].strip()
        return res

    assert(len(res)==1)
    return res[0]


def remove_page_number(txt):
    
    patterns = [
        r'\n\n\d+ \n\n',           # 49
        r'\n\n\*\*\d+\*\* \n\n',   # **247**
        r'\n\nPage \d+ \n\n',       # Page 123
        r'\n\n_\d+_ \n\n',   # _2470_ 
        r'```\n\n\d+\n\n```',
        
    ]

    for pattern in patterns:
        txt = re.sub(pattern, '\n\n', txt)
        
    return txt

def remove_fig_placeholders(txt):
    res = re.sub(r"\*\*==> picture \[\d+ x \d+\] intentionally omitted <==\*\*", '', txt)
    return res

# def remove_fig_txt(txt):
#     res = re.sub(r"\*\*----- Start of picture text -----\*\*<br>.+\*\*----- End of picture text -----\*\*<br>", '', txt)
#     res = re.sub(r"\*\*----- Start of picture text -----\*\*<br>\n.+\*\*----- End of picture text -----\*\*<br>", '', txt)

#     return res

def remove_fig_txt(txt):
    res = re.sub(
        r"\*\*----- Start of picture text -----\*\*<br>.+?\*\*----- End of picture text -----\*\*<br>",
        '',
        txt,
        flags=re.DOTALL
    )
    return res


MISC_TO_RM_DICT = {
    "taln-2015-long-015":  [
        "22 _[ème]_ TRAITEMENT AUTOMATIQUE DES LANGUES NATURELLES, CAEN, 2015 ", "MARTIN GLEIZE, BRIGITTE GRAU ",
        "_22[ème] Traitement Automatique des Langues Naturelles, Caen, 2015_",
        "_Proceedings of the 53rd Annual Meeting of the Association for Computational Linguistics and the 7th International Joint Conference on Natural Language Processing_ , pages 939–949, Beijing, China, July 26-31, 2015. _⃝_ c 2015 Association for Computational Linguistics",
    ],
    "taln-2014-long-022" : [
        "EXTRACTION DE RELATIONS SÉMANTIQUES ", "_[O-L2.4]_ ",
        "_21[ème] Traitement Automatique des Langues Naturelles, Marseille, 2014_ ",
        "_Proceedings of COLING 2014, the 25th International Conference on Computational Linguistics: Technical Papers_ , pages 2184–2194, Dublin, Ireland, August 23-29 2014. ",
        
    ],
    "taln-2014-long-020" :[
        "GRAPHE DE VOISINAGE DISTRIBUTIONNEL ", 
        '_[O-L2.2]_ ', "V. CLAVEAU, E. KIJAK, O. FERRET ",
        "_21[ème] Traitement Automatique des Langues Naturelles, Marseille, 2014_ ",
        "_Proceedings of COLING 2014, the 25th International Conference on Computational Linguistics: Technical Papers_ , pages 709–720, Dublin, Ireland, August 23-29 2014. ",
        
    ],
    
    "taln-2013-long-002" : [ 
        "_￿_ c ATALA ",   "_TALN-RÉCITAL 2013, 17-21 Juin, Les Sables d’Olonne_",
        "_Proceedings of Recent Advances in Natural Language Processing_ , pages 649–658, Hissar, Bulgaria, 7-13 September 2013. ",
    ], 
    "taln-2013-long-009" : [ 
        "_￿_ c ATALA ",   "_TALN-RÉCITAL 2013, 17-21 Juin, Les Sables d’Olonne_",
        "_Proceedings of the 51st Annual Meeting of the Association for Computational Linguistics_ , pages 497–506, Sofia, Bulgaria, August 4-9 2013. _⃝_ c 2013 Association for Computational Linguistics ",
        
    ], 
    
    "taln-2010-long-040": ["BENOÎT SAGOT, GÉRALDINE WALTHER ", "LEXIQUE MORPHOLOGIQUE ET TRAITEMENTS DE SURFACE POUR LE PERSAN "],
    "taln-2010-long-012": [
        "R. BEAUFORT, S. ROEKHAUT, L.-A. COUGNON, C. FAIRON ", 
        "UNE APPROCHE HYBRIDE POUR LA NORMALISATION DES SMS",
        "_Proceedings of the 48th Annual Meeting of the Association for Computational Linguistics_ , pages 770–779, Uppsala, Sweden, 11-16 July 2010. _⃝_ c 2010 Association for Computational Linguistics ",
        "_TALN 2010, Montréal, 19–23 juillet 2010_ ",
        
    ],
    "taln-2010-long-001": [
        "HOLGER SCHWENK ", "ADAPTATION EN TRADUCTION AUTOMATIQUE STATISTIQUE ",
        "_TALN 2010, Montréal, 19–23 juillet 2010_ ",
    ],
    "taln-2010-long-025":  [
        "ÉVALUATION AUTOMATIQUE DE RÉSUMÉS AVEC ET SANS RÉFÉRENCE ",
        "TORRES-MORENO ET AL. ", 
        "_TALN 2010, Montréal, 19–23 juillet 2010_ ",
        "_Coling 2010: Poster Volume_ , pages 1059–1067, Beijing, August 2010 ",
    ],
    "taln-2010-long-040":  ["_TALN 2010, Montréal, 19–23 juillet 2010_ "],
    
    "taln-2007-poster-009": ["Lorraine GOEURIOT, Natalia GRABAR, Béatrice DAILLE "],
    "taln-2007-long-035": [
        "Catherine RECANATI, Nicoleta ROGOVSCHI ", "_TALN 2007, Toulouse, 5–8 juin 2007_ ",
        "\nEnchaînements verbaux – étude sur le temps et l’aspect "
    ],
    
    "taln-2007-long-029": [
        "Lionel NICOLAS, Jacques FARRÉ, Éric VILLEMONTE DE LA CLERGERIE ", "TALN 2007, Toulouse, 5–8 juin 2007",
        "\nConfondre le coupable : corrections d’un lexique suggérées par une grammaire ",
    ],
    
    "taln-2006-poster-015": ["REPRÉSENTATION THÉMATIQUE DES CONNAISSANCES PAR LES NOTIONS DE BASE", "TALN 2006, Leuven, 10-13 avril 2006", "ALAIN JOUBERT, MATHIEU LAFOURCADE, DIDIER SCHWAB " ],
    "taln-2005-long-021": [
        "_Yves Bestgen_", 
        "_Amélioration de la segmentation automatique des textes_ ",
        "_TALN 2005, Dourdan, 6-10 juin 2005_ ",
        "© 2006 Association for Computational Linguistics \n\nComputational Linguistics \n\nVolume 32, Number 1 ",
        "Improving Text Segmentation \n\nBestgen ",
        "Computational Linguistics \n\nVolume 32, Number 1 "

        "Computational Linguistics \n\nVolume 32, Number 1 "
    ],
    "taln-2005-long-013": ["_L. Danlos_ ", "_ILLIMP_ ", "_TALN 2005, Dourdan, 6-10 juin 2005_ "],
    
    "taln-2004-long-005": ["_Olivier Ferret_", "_Découvrir des sens de mots_ ", "_TALN 2004, Fès, 19-21 avril 2004_ "],
    "taln-2002-long-013": ["_Olivier Ferret_ ", "_Segmenter et structurer thématiquement des textes_ ", "_TALN 2002, Nancy, 24-27 juin 2002_ "],

    "taln-2002-long-023": [
        "_TALN 2002, Nancy, 24-27 juin 2002_ ", 
        "_Thierry Poibeau, Dominique Dutoit et Sophie Bizouard_ ",
        "_Évaluer l’acquisition semi-automatique de classes sémantiques_ ",
    ],
    "taln-2000-long-030": [
        "Conférence TALN 2000, Lausanne, 16-18 octobre 2000 ",
        "Fouille du Web pour la collecte d’Entités Nommées ",
        "EN_cand =*>@? - ACB:DFEG>H - I? - ACB - BKJ$ELNMPO (3) Énumération EN_cand Q,M EN_cand =Q$RSMN= _and or_ EN_cand (4)",
    ],
    
    "TAL-2014-Braud": ["164 TAL. Volume 55 – n _[◦]_ 1/2014 "],
    "F08-1586":    ["Copyright © 2008 ISCA Accepted after peer review of full paper \n\n- September 22 26, Brisbane Australia "],
    "F14-107": ["XXX[e] édition des Journées d’Études sur la Parole (JEP 2014) ", "Le Mans — 23-27 juin 2014"],
    "F14-16": [
        "XXX[e] édition des Journées d’Études sur la Parole (JEP 2014) ", "Le Mans — 23-27 juin 2014",
        "16_janvier_2014.json corpus_entier.json 17_janvier_2014.json 18_janvier_2014.json ... sujet_1.acoustique.json sujet_1.linguistique.json sujet_1.metadonnees.json sujet_2.acoustique.json sujet_2.linguistique.json sujet_2.metadonnees.json ... ",
    ],
    "F14-38": ["_INTERSPEECH 2015_ " , "Copyright © 2015 ISCA \n\n- September 6 10, 2015, Dresden, Germany "],
    "F14-51": ["XXX[e] édition des Journées d’Études sur la Parole (JEP 2014) \n\nLe Mans — 23-27 juin 2014 "],
    "F12-1007":    ["_Actes de la conférence conjointe JEP-TALN-RECITAL 2012, volume 1: JEP_ , pages 49–56, Grenoble, 4 au 8 juin 2012. _⃝_ c 2012 ATALA & AFCP",
                    "_INTERSPEECH 2012_ ",
                    "INTERSPEECH 2012 ISCA's 13[th] Annual Conference Portland, OR, USA September 9-13, 2012 \n\nISCA Archive \n\nhttp://www.isca-speech.org/archive ",
                   ],
    "F12-1083": [
        "_INTERSPEECH 2011_ ",
        "_Actes de la conférence conjointe JEP-TALN-RECITAL 2012, volume 1: JEP_ , pages 657–664, Grenoble, 4 au 8 juin 2012. _⃝_ c 2012 ATALA & AFCP ",
        "Copyright © 2011 ISCA \n\n28- 31 August 2011, Florence, Italy ",
    ],
    "F12-1092": ["_Actes de la conférence conjointe JEP-TALN-RECITAL 2012, volume 1: JEP_ , pages 731–738, Grenoble, 4 au 8 juin 2012. _⃝_ c 2012 ATALA & AFCP "],
    "F12-1098": ["_Actes de la conférence conjointe JEP-TALN-RECITAL 2012, volume 1: JEP_ , pages 779–786, Grenoble, 4 au 8 juin 2012. _⃝_ c 2012 ATALA & AFCP "],

    'TAL-2011-1-Sajous': [
        "Language Resources & Evaluation ",
        "TAL. Volume 52 – n°1/2011, pages 11 à 35 \n\n12 TAL. Volume 52 – n°1/2011 ",
        "Enrichissement de lexiques sémantiques ",
        "TAL. Volume 52 – n°1/2011 ",
        # "\d+ TAL. Volume 52 – n°1/2011 ",
        
    ],

    'TAL-2012-2-Kessler': [
        "TAL. Volume 52 – n˚ 2/2012, pages 57 à 86 \n\n58 TAL. Volume 52 – n˚ 2/2012 ",
        # r"Extraction de dates saillantes \d+",
        # r"\d+ TAL\. Volume 52 – n˚ 2/2012 "
        "TAL. Volume 52 – n˚ 2/2012 ",
        "_Proceedings of the 50th Annual Meeting of the Association for Computational Linguistics_ , pages 730–739, Jeju, Republic of Korea, 8-14 July 2012. _⃝_ c 2012 Association for Computational Linguistics ",
        
    ],

    'TAL-2014-Braud': [ "TAL. Volume 55 – n _[◦]_ 1/2014, pages 135 à 165 \n\n136 TAL. Volume 55 – n _[◦]_ 1/2014 "],
    "TAL-2013-Claveau": [
        "_Proceedings of COLING 2012: Technical Papers_ , pages 629–646, COLING 2012, Mumbai, December 2012. ",
        "TAL. Volume 54 – n° 1/2013, pages 13 à 45 \n\n14 TAL. Volume 54 – n° 1/2013 ",
        "Morphosémantique pour la RI biomédicale",
        "TAL. Volume 54 – n° 1/2013 ",
    ],
    "taln-2009-court-039":  ["_Jean-Leon Bouraoui, Nadine Vigouroux_"],
}

def keep_aside_refs(txt, docid, doc_lang, to_store_ref_path):
    # marker_dict = {
    #     'fr': ["## **Références** ", "## **BIBLIOGRAPHIE** ", "## **R ´EFERENCES´** ", "## **8. Bibliographie** " ],
    #     'en': [ "## **References** ", "## **6 Bibliography**" ],
    #     # "## **8. References** ", "## **6. References** " , "## **7. References** ", "## **5. References** " ,
    #     #  "## **6 Bibliography**"
    # } 

    def store_refs(matched_txt, to_store_ref_path):
        txt = matched_txt[0].strip()
        refs = '\n'.join(matched_txt[1:])

        with open(to_store_ref_path, 'wt', encoding = 'utf-8') as f:
            f.write(refs)
        return txt, refs
                
         
    en_bib_pattern = re.compile(r"(## \*\*(?:\d+\.?\s+)?(?:References|Bibliography)\*\*)", flags=re.IGNORECASE)
    fr_bib_pattern = re.compile(r"(## \*\*(?:\d+\.?\s+)?(?:Références|BIBLIOGRAPHIE|R ´EFERENCES´|Bibliographie)\*\*)", flags=re.IGNORECASE)
    
    if doc_lang == 'en':
        
        res = re.split(en_bib_pattern, txt)
        if len(res) > 1:
            assert(len(res) == 3)
            txt, refs = store_refs(res, to_store_ref_path)
                
            return txt, refs
        
        res = re.split(r"(## References)", txt)
        if len(res) > 1:
            assert(len(res) == 3)
            txt, refs = store_refs(res, to_store_ref_path)
                
            return txt, refs
            
        print(f"Unrecognised refs in {docid}, {doc_lang}")

    elif doc_lang == 'fr':
        res = re.split(fr_bib_pattern, txt)
        if len(res) > 1:
            assert(len(res) == 3)
            txt, refs = store_refs(res, to_store_ref_path)
            
            return txt, refs

        print(f"Unrecognised refs in {docid}, {doc_lang}")
    else:
        print("Unsupported language")
    
    return txt, None



# def keep_aside_tables(txt, store_fpath):
#     return



In [73]:

DATA_DIR = "/Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603"
MD_DIR = f"{DATA_DIR}/pdf2md"
STORE_DIR = f"{DATA_DIR}/processed_md_auto"




for lang in ['en', 'fr']:

    for docid in os.listdir(MD_DIR):
        if docid in ['.DS_Store']:
            continue

        md_path = f"{MD_DIR}/{docid}/document_{lang}.md"
        if not os.path.exists(md_path):
            print(f'File not found: {md_path}')
            continue
        md_txt = open(md_path, 'rt').read()
        md_txt = remove_hal_page(md_txt)
        
        md_txt = remove_fig_placeholders(md_txt)
        md_txt = remove_fig_txt(md_txt)
        md_txt = remove_page_number(md_txt)
        md_txt = unicodedata.normalize('NFC', md_txt)

        if docid in MISC_TO_RM_DICT.keys():
            
            for tmp in MISC_TO_RM_DICT[docid]:
                # print(docid, tmp)
                md_txt = re.sub( re.escape(tmp.strip()), '', md_txt)


        sub_store_dir =  f"{STORE_DIR}/{docid}"
        Path(sub_store_dir).mkdir(exist_ok = True, parents = True)
        
        to_store_ref_path = f"{STORE_DIR}/{docid}/document_{lang}_ref.md"
        process_md_txt, refs = keep_aside_refs(md_txt, docid, lang, to_store_ref_path)

        to_write = '\n\n'.join([l for l in process_md_txt.split('\n\n') if l.strip()])
        to_store_path = f"{STORE_DIR}/{docid}/document_{lang}.md"
        with open(to_store_path, 'wt', encoding = 'utf-8') as f:
            f.write(to_write)



        

File not found: /Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603/pdf2md/taln-2009-court-039/document_en.md


In [32]:
# run once

# STORE_DIR1 = f"{DATA_DIR}/processed_md"
# Path(STORE_DIR1).mkdir(exist_ok = True, parents = True)

# os.system(f"cp -r {STORE_DIR}/ {STORE_DIR1}")


# for docid in os.listdir(STORE_DIR1):
#     if docid in ['.DS_Store']:
#         continue

#     for lang in ['en', 'fr']:
#         author_path = f"{STORE_DIR1}/{docid}/document_{lang}_authors.md"
#         with open(author_path, 'wt') as f:
#             f.write('')

#         abstract_en_in_fr_path = f"{STORE_DIR1}/{docid}/abstract_en_in_fr.md"
#         with open(author_path, 'wt') as f:
#             f.write('')



In [83]:
import re

def extract_abstract_and_keywords(text):
    result = {}
    
    # Extract English Abstract
    abstract_match = re.search(
        r'\*\*Abstract\.\*\*\s*(.*?)(?=\n\n|\*\*)', 
        text, re.DOTALL
    )
    if abstract_match:
        result['abstract'] = abstract_match.group(1).strip()
    
    # Extract English Keywords
    keywords_match = re.search(
        r'\*\*Keywords:\*\*\s*(.*?)(?=\n\n|$)', 
        text, re.DOTALL
    )
    if keywords_match:
        result['keywords'] = keywords_match.group(1).strip()
        
    return result


def extract_abstract_en_in_fr(fr_txt, to_store_path):
    match = re.search(r'(## ABSTRACT.*(?:KEYWORDS|MOTS-CLÉS)[^\n]*)', fr_txt, re.DOTALL | re.IGNORECASE)
    if match:
        abstract = match.group(1)
        if not os.path.exists(to_store_path):
            with open(to_store_path, 'wt', encoding = 'utf-8') as f:
                f.write(abstract)
 
            fr_txt = fr_txt.replace(abstract, '')
    else:
        res = extract_abstract_and_keywords(fr_txt)

        if len(res)>0:
            abstract = "**Abstract.** " + res.get('abstract', '') 
            keywords = '\n\n**Keywords:** '+ res.get('keywords', '') 
            to_write = (abstract + keywords).strip()
            
            # if not os.path.exists(to_store_path):
            if True:
                
                with open(to_store_path, 'wt', encoding = 'utf-8') as f:
                    f.write(to_write)
                fr_txt = fr_txt.replace(abstract, '') 
                fr_txt = fr_txt.replace(keywords, '')    
                
    
    return fr_txt



Abstract: The main objective of our study consists to characterise the comparability of corpora, and we address this issue in the monolingual context through the disctinction of expert and non expert documents.
Keywords: corpus linguistics, comparable corpora, learning algorithms, stylistic analysis, degree of comparability.


In [100]:


STORE_DIR1 = f"{DATA_DIR}/processed_md"

for docid in os.listdir(STORE_DIR1):
    if docid in ['.DS_Store']:
        continue

    print(docid)
    for lang in ['fr']:
        md_path = f"{STORE_DIR1}/{docid}/document_{lang}.md"
        md_txt = open(md_path, 'rt').read()
        md_txt = remove_fig_txt(md_txt)
        md_txt = remove_page_number(md_txt)
        
        abstract_en_in_fr_path = f"{STORE_DIR1}/{docid}/abstract_en_in_fr.md"
        process_md_txt = extract_abstract_en_in_fr(md_txt, abstract_en_in_fr_path)

        to_write = '\n\n'.join([l for l in process_md_txt.split('\n\n') if l.strip()])
        with open(md_path, 'wt', encoding = 'utf-8') as f:
            f.write(to_write)
    

In [44]:
# ! pip install ftfy
# import ftfy

# fixed = ftfy.fix_text("rˆole")
# print(fixed)
# return rˆole , doesn't work

In [42]:
import re

def move_footnotes_to_end(text):
    footnotes = []
    
    def collect(match):
        footnotes.append(match.group(0).strip())
        return ''  # remove from original position
    
    # Match one or more consecutive "> ..." lines (multi-line footnotes too)
    pattern = r'(?:^>.*\n?)+'
    
    cleaned = re.sub(pattern, collect, text, flags=re.MULTILINE)
    
    # Clean up extra blank lines left behind
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned).strip()
    
    if footnotes:
        cleaned += '\n\n' + '\n\n'.join(footnotes)
    
    return cleaned

def fix_latex_accents(text):
    replacements = {
        'rˆole': 'rôle',
        # General pattern for circumflex
        'aˆ': 'â', 'eˆ': 'ê', 'iˆ': 'î', 'oˆ': 'ô', 'uˆ': 'û',
        'Aˆ': 'Â', 'Eˆ': 'Ê', 'Iˆ': 'Î', 'Oˆ': 'Ô', 'Uˆ': 'Û',
        '`e': 'è', '´e': 'é', '`a': 'à', '`u': 'ù', 'ˆe': 'ê',
        'ˆa': 'â', 'ˆi': 'î', ' ´E': 'É', '�': '',
        
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text

import re

def remove_special_symbol_stricks(text):
    # text ="~~_n_~~ _bEnfant_"
    res = re.sub(r'~~_?([^~_]+?)_?~~\s*_?([^_]+?)_?(?=\s|$|\))', r'\1\2', text)
    res = re.sub(r'. . .', '...', res)
    return res




STORE_DIR1 = f"{DATA_DIR}/processed_md"

for docid in os.listdir(STORE_DIR1):
    if docid in ['.DS_Store']:
        continue

    print(docid)
    for lang in ['en', 'fr']:
        md_path = f"{STORE_DIR1}/{docid}/document_{lang}.md"

        if not os.path.exists(md_path):
            print(f"File not found : {md_path}")
            continue
            
        md_txt = open(md_path, 'rt').read()
        # md_txt = move_footnotes_to_end(md_txt)
        # md_txt = remove_fig_txt(md_txt)
        # md_txt = fix_latex_accents(md_txt)
        md_txt = remove_special_symbol_stricks(md_txt)
        
        to_write = md_txt
        with open(md_path, 'wt', encoding = 'utf-8') as f:
            f.write(to_write)

F04-25
TAL-2012-2-Kessler
F14-3
F04-46
taln-2007-long-029
F12-1092
TAL-2013-Claveau
taln-2005-long-013
TAL-2014-Braud
taln-2002-long-023
TAL-2011-1-Sajous
taln-2002-long-013
taln-2013-long-009
taln-2006-poster-015
taln-2015-long-015
taln-2014-long-022
taln-2007-long-035
F14-38
taln-2000-long-030
F12-1007
F12-1098
F14-16
F14-107
taln-2010-long-040
F08-1586
taln-2009-court-039
File not found : /Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603/processed_md/taln-2009-court-039/document_en.md
taln-2010-long-025
taln-2007-poster-009
taln-2005-long-021
taln-2010-long-012
taln-2010-long-001
taln-2004-long-005
taln-2014-long-020
F14-51
F12-1083
taln-2013-long-002


## extract tables

In [2]:
import re


def extract_blocks(blocks: list[str]) -> dict:
    """
    Split markdown by double newlines and extract:
      - Table blocks: contain a separator row (|---|---|)
      - Heading blocks: start with # or ##
    Returns extracted items with their block positions and the remaining blocks.
    """
    # blocks = markdown_text.split("\n\n")
    table_pattern   = re.compile(r"\|[-:| ]+\|")
    heading_pattern = re.compile(r"^#{1,2} .+", re.MULTILINE)

    tables   = {}  # (block_id, text)
    headings = {}  # (block_id, text)
    txt_dict = {} # (block_id, text)

    for i, block in enumerate(blocks):
        if table_pattern.search(block):
            tables[i] = block
        elif heading_pattern.match(block.strip()):
            headings[i]= block
            txt_dict[i]= block
        else:
            txt_dict[i]= block

    return {"tables": tables, "headings": headings, "txt": txt_dict}






In [4]:

DATA_DIR = "/Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603"
STORE_DIR1 = f"{DATA_DIR}/processed_md"
TXT_DIR = f"{DATA_DIR}/processed_md2txt"

for docid in os.listdir(STORE_DIR1):
    if docid in ['.DS_Store']:
        continue

    sub_store_dir = f"{TXT_DIR}/{docid}/table_heading_misc"
    Path(sub_store_dir).mkdir(exist_ok = True, parents = True)

    print(docid)
    for lang in ['en', 'fr']:
        md_path = f"{STORE_DIR1}/{docid}/document_{lang}.md"
        if not os.path.exists(md_path):
            print(f"File not found : {md_path}")
            continue            
        md_txt = [l.strip() for l in open(md_path, 'rt').read().split('\n\n') if l.strip()]

        if md_txt[0][:2] == '# ' and md_txt[1][:2] == '# ':
            title = md_txt[0].strip() + ' ' + md_txt[1][2:].strip()
            md_txt = [title] + md_txt[2:]
        
        extracted = extract_blocks(md_txt) 

        # store tables
        table_store_path = f"{sub_store_dir}/tables_{lang}.tsv"
        pd.DataFrame(extracted['tables'], index = ['table']).T.to_csv(table_store_path, sep = '\t')

        section_store_path = f"{sub_store_dir}/headings_{lang}.tsv"
        pd.DataFrame(extracted['headings'], index = ['heading']).T.to_csv(section_store_path, sep = '\t')

        
        to_write = []
        for i in sorted(extracted['txt'].keys()):
            # quite slow, to optimize
            t = pypandoc.convert_text(extracted['txt'][i], 'plain', format='md',  extra_args=['--wrap=none'])
            t = unicodedata.normalize('NFC', t)
            to_write.append(t.strip())

           

        txt_store_path =  f"{TXT_DIR}/{docid}/document_{lang}.txt"
        with open(txt_store_path, 'wt', encoding = 'utf-8' ) as f:
            f.write('\n\n'.join(to_write))
        

F04-25
TAL-2012-2-Kessler
F14-3
F04-46
taln-2007-long-029
F12-1092
TAL-2013-Claveau
taln-2005-long-013
TAL-2014-Braud
taln-2002-long-023
TAL-2011-1-Sajous
taln-2002-long-013
taln-2013-long-009
taln-2006-poster-015
taln-2015-long-015
taln-2014-long-022
taln-2007-long-035
F14-38
taln-2000-long-030
F12-1007
F12-1098
F14-16
F14-107
taln-2010-long-040
F08-1586
taln-2009-court-039
File not found : /Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603/processed_md/taln-2009-court-039/document_en.md
taln-2010-long-025
taln-2007-poster-009
taln-2005-long-021
taln-2010-long-012
taln-2010-long-001
taln-2004-long-005
taln-2014-long-020
F14-51
F12-1083
taln-2013-long-002


## make prompt zero-shot

In [8]:
import re
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import string
import json


# make prompts according to TowerEval-Data-v0.1
def make_prompt_MT(src, src_lang = 'English', tgt_lang = 'French'):
    template_string = f"Translate the following $src_lang source text to $tgt_lang:\n$src_lang: $src\n$tgt_lang: "
    template = string.Template( template_string)
    prompt = template.safe_substitute(src = src, src_lang = src_lang, tgt_lang = tgt_lang)
    return prompt


def make_prompt_APE(src, sys,  src_lang = 'English', tgt_lang = 'French'):
    template_string = f"The primary goal of automatic post-editing is to boost the translation's quality by making small adjustments to address any existing errors in the translation. If the translation is already accurate, there is no need to change it and instead it should be copied.\nSource ($src_lang): $src\nTranslation ($tgt_lang): $sys\nPost-edited: "
    template = string.Template( template_string)
    prompt = template.safe_substitute(src = src, sys = sys, src_lang = src_lang, tgt_lang = tgt_lang)
    return prompt


def make_prompts_corpus(align_dir, root_store_dir, extracted_align_dir, src_lang = 'English', tgt_lang = 'French'):
    # STORE_DIR = f"{ROOT_PATH}/instructions_mt_ape_fr_en"
    Path(root_store_dir).mkdir(parents=True, exist_ok=True)
    print(f"Making prompts using alignement from {align_dir}, storing to {root_store_dir}")

    for d in os.listdir(align_dir):
        if d in ['.DS_Store', 'taln-2009-court-039']:
            continue
        print(d)
        
        # align_df = pd.read_csv(  f"{align_dir}/{d}/align_{l1}-{l2}.tsv", sep = '\t', index_col = 0 )
        align_df = pd.read_csv(  f"{align_dir}/{d}/align_fr-en.tsv", sep = '\t', index_col = 0 )
        align_df = align_df.rename(columns = { 'fr': 'French', 'en': 'English'})
        
        
        # extract alignments with non-empty source segments
        tmp_df = align_df.loc[align_df[src_lang].dropna().index][[src_lang, tgt_lang]].T
        
        # MT
        mt_prompts = tmp_df.apply(lambda x : make_prompt_MT(x[src_lang], src_lang = src_lang, tgt_lang = tgt_lang) ).to_list()
        
        store_dir = f"{root_store_dir}/{d}/mt"
        Path(store_dir).mkdir(parents=True, exist_ok=True) 
        with open(f"{store_dir}/instructions.json", 'wt',  encoding = 'utf-8') as f:
            json.dump(mt_prompts, f)
            
        
        # APE
        ape_prompts = tmp_df.apply(lambda x : make_prompt_APE(x[src_lang], x[tgt_lang], src_lang = src_lang, tgt_lang = tgt_lang) ).to_list()
        
        store_dir = f"{root_store_dir}/{d}/ape"
        Path(store_dir).mkdir(parents=True, exist_ok=True) 
        with open(f"{store_dir}/instructions.json", 'wt',  encoding = 'utf-8') as f:
            json.dump(ape_prompts, f)

        # en & fr
        store_dir = f"{extracted_align_dir}/{d}/non_empty_{src_lang}"
        Path(store_dir).mkdir(parents=True, exist_ok=True) 
        with open(f"{store_dir}/alignments.en", 'wt',  encoding = 'utf-8') as f:
            f.write('\n'.join(tmp_df.T.English.fillna('nan').to_list()))
        
        with open(f"{store_dir}/alignments.fr", 'wt',  encoding = 'utf-8') as f:
            f.write('\n'.join(tmp_df.T.French.fillna('nan').to_list()))

        
if True:
    DATA_DIR = "/Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603"
    align_dir = f"{DATA_DIR}/segmented_aligned_txt_16"
    STORE_DIR = f"{DATA_DIR}/ACL-v3-2026-03"
    
    
    extracted_align_dir = f"{STORE_DIR}/extracted_alignments"

    root_store_dir = f"{STORE_DIR}/instructions_mt_ape_en_fr"
    make_prompts_corpus(align_dir, root_store_dir, extracted_align_dir, src_lang = 'English', tgt_lang = 'French')

    root_store_dir = f"{STORE_DIR}/instructions_mt_ape_fr_en"
    make_prompts_corpus(align_dir, root_store_dir, extracted_align_dir, src_lang = 'French', tgt_lang = 'English')





Making prompts using alignement from /Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603/segmented_aligned_txt_16, storing to /Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603/ACL-v3-2026-03/instructions_mt_ape_en_fr
F04-25
TAL-2012-2-Kessler
F14-3
F04-46
taln-2007-long-029
F12-1092
TAL-2013-Claveau
taln-2005-long-013
TAL-2014-Braud
taln-2002-long-023
TAL-2011-1-Sajous
taln-2002-long-013
taln-2013-long-009
taln-2006-poster-015
taln-2015-long-015
taln-2014-long-022
taln-2007-long-035
F14-38
taln-2000-long-030
F12-1007
F12-1098
F14-16
F14-107
taln-2010-long-040
F08-1586
taln-2010-long-025
taln-2007-poster-009
taln-2005-long-021
taln-2010-long-012
taln-2010-long-001
taln-2004-long-005
taln-2014-long-020
F14-51
F12-1083
taln-2013-long-002
Making prompts using alignement from /Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603/segmented_aligned_txt_16, storing to /Users/ziqianpeng/MaTOS/data/TAL/corpora/ACL_update202603/ACL-v3-2026-03/instructions_mt_ape_fr_en
F04-25
TAL